# EDA-04: DV Block Graph Connectivity

Extends the block analysis from EDA-03 by:

1. **Including trivial blocks** — single-column values with row count ≥ 2 that
   are not shared with any other column.
2. **Removing inner blocks** — blocks whose row set $R_b$ is strictly contained
   in another block's row set ($R_b \subsetneq R_a$).
3. **Building a graph** — remaining blocks as vertices (weighted by $HC_b$),
   edges between blocks with **normal overlap**
   ($R_a \cap R_b \neq \emptyset$, neither is a subset of the other).
4. **Computing connectivity** — connected components of the graph.

By construction, after removing inner blocks all block pairs have either
no overlap or normal overlap — strict containment is impossible.

In [1]:
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from llm_sql_phc_eval.datasets import load_datasets

DATA_DIR = Path("/Users/owhite/dev/whitechno-github/llm-sql-phc-databases")
datasets = load_datasets(DATA_DIR)
movies, products, beer, pdmx, fever, squad, bird = (
    datasets[k] for k in ("Movies", "Products", "Beer", "PDMX", "FEVER", "SQuAD", "BIRD")
)
print("Loaded:", {k: len(v) for k, v in datasets.items()})


Loaded: {'Movies': 15000, 'Products': 14890, 'Beer': 28479, 'PDMX': 10000, 'FEVER': 145449, 'SQuAD': 87599, 'BIRD': 15000}


## Functions

In [2]:
# ── helpers ────────────────────────────────────────────────────────────────────

def _is_hashable_col(series, n_sample=200):
    for v in series.dropna().iloc[:n_sample]:
        if isinstance(v, (list, dict)):
            return False
    return True


def find_all_blocks(df):
    """
    Find all blocks (multi-column AND trivial single-column) with row_count >= 2.

    Returns list of block dicts, each with:
      sig (frozenset), row_count, n_cols, cols, hc, entries, trivial (bool)
    """
    df = df.reset_index(drop=True)
    sig_groups = defaultdict(list)

    for col in df.columns:
        if not _is_hashable_col(df[col]):
            continue
        for val, idx_arr in df.groupby(col, sort=False).indices.items():
            n = len(idx_arr)
            if n < 2:
                continue
            sig = frozenset(idx_arr.tolist())
            hc  = len(str(val)) ** 2 * (n - 1)
            sig_groups[sig].append((col, val, hc))

    blocks = []
    for sig, entries in sig_groups.items():
        cols_in_block = sorted({col for col, val, hc in entries})
        blocks.append({
            "sig":       sig,
            "row_count": len(sig),
            "n_cols":    len(cols_in_block),
            "cols":      cols_in_block,
            "hc":        sum(hc for col, val, hc in entries),
            "entries":   sorted(entries, key=lambda x: x[0]),
            "trivial":   len(cols_in_block) == 1,
        })
    return blocks


def filter_inner_blocks(blocks):
    """
    Remove blocks whose row set is strictly contained in another block's row set.

    Uses a row→blocks mapping so each block is only checked against blocks that
    share at least one row with it, making this much faster than O(n²).
    """
    if not blocks:
        return []

    # row → indices of blocks containing that row
    row_to_blocks = defaultdict(list)
    for i, b in enumerate(blocks):
        for r in b["sig"]:
            row_to_blocks[r].append(i)

    is_inner = [False] * len(blocks)
    for j, b in enumerate(blocks):
        if is_inner[j]:
            continue
        rc_j = b["row_count"]
        sig_j = b["sig"]
        # Candidate parents: blocks that share the first row and are strictly larger
        first_row = next(iter(sig_j))
        for i in row_to_blocks[first_row]:
            if i == j or is_inner[i]:
                continue
            if blocks[i]["row_count"] > rc_j and sig_j.issubset(blocks[i]["sig"]):
                is_inner[j] = True
                break

    return [b for b, inner in zip(blocks, is_inner) if not inner]


def _compute_inner_hc(all_blocks, kept):
    """
    For each kept block, sum the HC of all inner-removed blocks it contains.

    Each inner block contributes its HC to every kept block that strictly
    contains it (R_inner ⊊ R_kept). No tightest-parent logic: if C ⊂ B ⊂ A
    and both B and A are kept, HC_C is added to both IHC_B and IHC_A.
    Only raw HC (never IHC) is propagated, so there is no double-counting.
    """
    kept_ids     = {id(b) for b in kept}
    inner_blocks = [b for b in all_blocks if id(b) not in kept_ids]

    if not inner_blocks:
        return [(0, 0)] * len(kept)

    # row → kept indices, for fast candidate lookup
    row_to_kept = defaultdict(list)
    for i, b in enumerate(kept):
        for r in b["sig"]:
            row_to_kept[r].append(i)

    result = [0] * len(kept)
    counts = [0] * len(kept)
    for ib in inner_blocks:
        first_row = next(iter(ib["sig"]))
        for i in row_to_kept[first_row]:
            kb = kept[i]
            if kb["row_count"] > ib["row_count"] and ib["sig"].issubset(kb["sig"]):
                result[i] += ib["hc"]
                counts[i] += 1

    return list(zip(result, counts))


def find_edges(blocks):
    """
    Find all pairs of blocks with normal overlap (non-empty row intersection).

    After filter_inner_blocks(), no strict containment remains, so any shared
    row implies normal overlap. Uses a row→blocks mapping for efficiency.
    """
    row_to_blocks = defaultdict(list)
    for i, b in enumerate(blocks):
        for r in b["sig"]:
            row_to_blocks[r].append(i)

    edge_set = set()
    for peers in row_to_blocks.values():
        for k in range(len(peers)):
            for l in range(k + 1, len(peers)):
                i, j = peers[k], peers[l]
                edge_set.add((min(i, j), max(i, j)))
    return list(edge_set)


# ── union-find for connected components ────────────────────────────────────────

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, x, y):
        self.parent[self.find(x)] = self.find(y)


def graph_components(n_vertices, edges):
    """Return list of components (each a list of vertex indices), largest first."""
    uf = UnionFind(n_vertices)
    for i, j in edges:
        uf.union(i, j)
    comp = defaultdict(list)
    for v in range(n_vertices):
        comp[uf.find(v)].append(v)
    return sorted(comp.values(), key=len, reverse=True)


# ── full pipeline ──────────────────────────────────────────────────────────────

def analyze_dataset(df, name):
    t0 = time.time()

    all_blocks  = find_all_blocks(df)
    n_total     = len(all_blocks)
    n_trivial   = sum(1 for b in all_blocks if b["trivial"])

    kept        = filter_inner_blocks(all_blocks)
    n_inner     = n_total - len(kept)

    # attach inner_hc to each kept block
    for b, (ihc, n_inners) in zip(kept, _compute_inner_hc(all_blocks, kept)):
        b["inner_hc"] = ihc
        b["n_inners"] = n_inners

    edges       = find_edges(kept)
    components  = graph_components(len(kept), edges)

    # sort components by total HC descending
    components  = sorted(
        components,
        key=lambda c: sum(kept[i]["hc"] for i in c),
        reverse=True,
    )

    elapsed = time.time() - t0
    return {
        "name":        name,
        "n_total":     n_total,
        "n_trivial":   n_trivial,
        "n_inner":     n_inner,
        "n_vertices":  len(kept),
        "n_edges":     len(edges),
        "n_components": len(components),
        "largest_comp": max((len(c) for c in components), default=0),
        "blocks":      kept,
        "edges":       edges,
        "components":  components,
        "elapsed":     elapsed,
    }


def show_analysis(result, top_n_comps=5, top_n_blocks=10):
    """Print graph stats and top components for one dataset."""
    name       = result["name"]
    blocks     = result["blocks"]
    components = result["components"]

    print(f"{'─'*55}")
    print(f"{name}  ({result['n_total']:,} blocks total, {result['elapsed']:.1f}s)")
    print(f"  trivial (1-col)       : {result['n_trivial']:,}")
    print(f"  inner removed         : {result['n_inner']:,}")
    print(f"  vertices (kept)       : {result['n_vertices']:,}")
    print(f"  edges (normal overlap): {result['n_edges']:,}")
    print(f"  connected components  : {result['n_components']:,}")

    if not components:
        return

    size_dist = dict(sorted(Counter(len(c) for c in components).items(), reverse=True))
    print(f"  component size dist   : {size_dist}")
    print()

    # precompute degree per vertex index
    degrees = Counter(v for e in result["edges"] for v in e)

    n_show = min(top_n_comps, len(components))
    print(f"  Top {n_show} components by total HC:")
    for rank, comp in enumerate(components[:top_n_comps], 1):
        comp_blocks = [blocks[j] for j in comp]
        total_hc    = sum(b["hc"] for b in comp_blocks)
        total_rows  = len({r for b in comp_blocks for r in b["sig"]})
        max_rc      = max(b["row_count"] for b in comp_blocks)
        max_deg     = max((degrees[j] for j in comp), default=0)
        print(f"\n  [{rank}] {len(comp)} blocks  total_hc={total_hc:,}  "
              f"total_rows={total_rows:,}  max_rows={max_rc}  max_degree={max_deg}")
        def _fmt_block(b, j):
            parts = []
            for col, val, h in b["entries"][:2]:
                s = str(val)
                s = s[:22] + "..." if len(s) > 25 else s
                parts.append(f"{col}={s!r}")
            if len(b["entries"]) > 2:
                extra = ", ".join(col for col, val, h in b["entries"][2:])
                parts.append(f"+[{extra}]")
            return (f"    rows={b['row_count']:,}  n_cols={b['n_cols']}  deg={degrees[j]}  "
                    f"hc={b['hc']:,}  inner_hc={b['inner_hc']:,}  n_inners={b['n_inners']:,}  {' | '.join(parts)}")

        print(f"  top {min(top_n_blocks, len(comp))} by hc:")
        for b, j in sorted(
            zip(comp_blocks, comp), key=lambda x: x[0]["hc"], reverse=True
        )[:top_n_blocks]:
            print(_fmt_block(b, j))

        print(f"  top {min(top_n_blocks, len(comp))} by inner_hc:")
        for b, j in sorted(
            zip(comp_blocks, comp), key=lambda x: x[0]["inner_hc"], reverse=True
        )[:top_n_blocks]:
            print(_fmt_block(b, j))

        print(f"  top {min(top_n_blocks, len(comp))} by hc + inner_hc:")
        for b, j in sorted(
            zip(comp_blocks, comp), key=lambda x: x[0]["hc"] + x[0]["inner_hc"], reverse=True
        )[:top_n_blocks]:
            print(_fmt_block(b, j))

## Run on all datasets

In [3]:
all_results = {}
for ds_name, df in datasets.items():
    result = analyze_dataset(df, ds_name)
    all_results[ds_name] = result
    print(f"{ds_name:10s}: {result['n_total']:5,} blocks → "
          f"{result['n_vertices']:4,} vertices  "
          f"{result['n_edges']:5,} edges  "
          f"{result['n_components']:4,} components  "
          f"({result['elapsed']:.1f}s)")

# Combined summary table
rows = []
for ds_name, r in all_results.items():
    rows.append({
        "dataset":      ds_name,
        "n_total":      r["n_total"],
        "n_trivial":    r["n_trivial"],
        "n_inner":      r["n_inner"],
        "n_vertices":   r["n_vertices"],
        "n_edges":      r["n_edges"],
        "n_components": r["n_components"],
        "largest_comp": r["largest_comp"],
    })
display(pd.DataFrame(rows).set_index("dataset"))


Movies    : 1,581 blocks →  294 vertices  2,322 edges     1 components  (0.2s)
Products  : 3,120 blocks →  474 vertices  2,817 edges     1 components  (0.2s)
Beer      : 11,193 blocks → 5,519 vertices  151,853 edges     1 components  (0.7s)
PDMX      : 10,105 blocks →    1 vertices      0 edges     1 components  (0.5s)
FEVER     : 5,898 blocks →  678 vertices  1,352 edges     1 components  (0.9s)
SQuAD     : 26,599 blocks → 24,807 vertices  25,430 edges  5,714 components  (0.9s)
BIRD      : 3,509 blocks → 3,509 vertices     62 edges  3,448 components  (0.1s)


,n_total,n_trivial,n_inner,n_vertices,n_edges,n_components,largest_comp
dataset,,,,,,,
Movies,1581,331,1287,294,2322,1,294
Products,3120,1059,2646,474,2817,1,474
Beer,11193,6603,5674,5519,151853,1,5519
PDMX,10105,9266,10104,1,0,1,1
FEVER,5898,5897,5220,678,1352,1,678
SQuAD,26599,26539,1792,24807,25430,5714,18209
BIRD,3509,38,0,3509,62,3448,6


## Movies

In [4]:
show_analysis(all_results["Movies"])

───────────────────────────────────────────────────────
Movies  (1,581 blocks total, 0.2s)
  trivial (1-col)       : 331
  inner removed         : 1,287
  vertices (kept)       : 294
  edges (normal overlap): 2,322
  connected components  : 1
  component size dist   : {294: 1}

  Top 1 components by total HC:

  [1] 294 blocks  total_hc=58,118,098  total_rows=15,000  max_rows=11266  max_degree=292
  top 10 by hc:
    rows=29  n_cols=5  deg=4  hc=7,214,872  inner_hc=0  n_inners=0  genres='Action & Adventure, Ho...' | movieinfo='In a society ravaged b...' | +[movietitle, productioncompany, rottentomatoeslink]
    rows=18  n_cols=5  deg=5  hc=4,125,679  inner_hc=0  n_inners=0  genres='Classics, Comedy, Dram...' | movieinfo='After Regina Lampert (...' | +[movietitle, productioncompany, rottentomatoeslink]
    rows=17  n_cols=5  deg=5  hc=4,035,968  inner_hc=0  n_inners=0  genres='Documentary, Drama, Mu...' | movieinfo='Individuals from all w...' | +[movietitle, productioncompany, rottentom

## Products

In [5]:
show_analysis(all_results["Products"])

───────────────────────────────────────────────────────
Products  (3,120 blocks total, 0.2s)
  trivial (1-col)       : 1,059
  inner removed         : 2,646
  vertices (kept)       : 474
  edges (normal overlap): 2,817
  connected components  : 1
  component size dist   : {474: 1}

  Top 1 components by total HC:

  [1] 474 blocks  total_hc=5,375,985,117  total_rows=14,890  max_rows=13591  max_degree=472
  top 10 by hc:
    rows=34  n_cols=3  deg=8  hc=1,136,385,525  inner_hc=0  n_inners=0  description='View larger Braun Silk...' | parent_asin='B00AX3YNPC' | +[product_title]
    rows=50  n_cols=2  deg=9  hc=685,208,258  inner_hc=4,800  n_inners=2  description='Product Description Im...' | product_title='Philips Sonicare HX335...'
    rows=20  n_cols=3  deg=8  hc=304,351,595  inner_hc=0  n_inners=0  description='Product Description No...' | parent_asin='B000A23CQM' | +[product_title]
    rows=125  n_cols=3  deg=13  hc=202,781,168  inner_hc=0  n_inners=0  description="Whether you're tryi

## Beer

In [6]:
show_analysis(all_results["Beer"])

───────────────────────────────────────────────────────
Beer  (11,193 blocks total, 0.7s)
  trivial (1-col)       : 6,603
  inner removed         : 5,674
  vertices (kept)       : 5,519
  edges (normal overlap): 151,853
  connected components  : 1
  component size dist   : {5519: 1}

  Top 1 components by total HC:

  [1] 5519 blocks  total_hc=12,560,948  total_rows=28,479  max_rows=13229  max_degree=5165
  top 10 by hc:
    rows=1,805  n_cols=1  deg=2213  hc=1,414,336  inner_hc=806,388  n_inners=283  beer/style='India Pale Ale &#40;IP...'
    rows=463  n_cols=1  deg=772  hc=443,982  inner_hc=145,395  n_inners=80  beer/style='Belgian White &#40;Wit...'
    rows=1,217  n_cols=1  deg=1674  hc=393,984  inner_hc=574,173  n_inners=195  beer/style='Belgian Strong Ale'
    rows=985  n_cols=1  deg=1415  hc=355,224  inner_hc=659,392  n_inners=163  beer/style='Imperial/Double IPA'
    rows=733  n_cols=1  deg=1041  hc=292,800  inner_hc=156,975  n_inners=121  beer/style='Golden Ale/Blond Ale'
    

## PDMX

In [7]:
show_analysis(all_results["PDMX"])

───────────────────────────────────────────────────────
PDMX  (10,105 blocks total, 0.5s)
  trivial (1-col)       : 9,266
  inner removed         : 10,104
  vertices (kept)       : 1
  edges (normal overlap): 0
  connected components  : 1
  component size dist   : {1: 1}

  Top 1 components by total HC:

  [1] 1 blocks  total_hc=1,109,889  total_rows=10,000  max_rows=10000  max_degree=0
  top 1 by hc:
    rows=10,000  n_cols=6  deg=0  hc=1,109,889  inner_hc=102,091,798  n_inners=10,104  hasannotations='True' | isdraft='False' | +[isofficial, isuserpublisher, publisher, subsetall]
  top 1 by inner_hc:
    rows=10,000  n_cols=6  deg=0  hc=1,109,889  inner_hc=102,091,798  n_inners=10,104  hasannotations='True' | isdraft='False' | +[isofficial, isuserpublisher, publisher, subsetall]
  top 1 by hc + inner_hc:
    rows=10,000  n_cols=6  deg=0  hc=1,109,889  inner_hc=102,091,798  n_inners=10,104  hasannotations='True' | isdraft='False' | +[isofficial, isuserpublisher, publisher, subsetall]


## FEVER

Note: list-valued `evidence` column is skipped.

In [8]:
show_analysis(all_results["FEVER"])

───────────────────────────────────────────────────────
FEVER  (5,898 blocks total, 0.9s)
  trivial (1-col)       : 5,897
  inner removed         : 5,220
  vertices (kept)       : 678
  edges (normal overlap): 1,352
  connected components  : 1
  component size dist   : {678: 1}

  Top 1 components by total HC:

  [1] 678 blocks  total_hc=31,963,324  total_rows=145,449  max_rows=109810  max_degree=676
  top 10 by hc:
    rows=35,639  n_cols=2  deg=676  hc=15,003,598  inner_hc=1,994,162  n_inners=657  label='NOT ENOUGH INFO' | verifiable='NOT VERIFIABLE'
    rows=109,810  n_cols=1  deg=676  hc=10,980,900  inner_hc=29,168,228  n_inners=4,563  verifiable='VERIFIABLE'
    rows=11  n_cols=1  deg=2  hc=466,560  inner_hc=0  n_inners=0  claim='Beverly Hills, 90210 a...'
    rows=6  n_cols=1  deg=2  hc=383,645  inner_hc=0  n_inners=0  claim='Drama school offers a ...'
    rows=8  n_cols=1  deg=2  hc=344,988  inner_hc=0  n_inners=0  claim='Laura Linney was in Lo...'
    rows=9  n_cols=1  deg=2  h

## SQuAD

In [9]:
show_analysis(all_results["SQuAD"])

───────────────────────────────────────────────────────
SQuAD  (26,599 blocks total, 0.9s)
  trivial (1-col)       : 26,539
  inner removed         : 1,792
  vertices (kept)       : 24,807
  edges (normal overlap): 25,430
  connected components  : 5,714
  component size dist   : {18209: 1, 16: 1, 14: 1, 9: 4, 7: 6, 6: 9, 5: 35, 4: 25, 3: 259, 2: 11, 1: 5362}

  Top 5 components by total HC:

  [1] 18209 blocks  total_hc=32,707,359,140  total_rows=61,135  max_rows=233  max_degree=218
  top 10 by hc:
    rows=8  n_cols=1  deg=2  hc=55,824,832  inner_hc=0  n_inners=0  context='There is a very active...'
    rows=14  n_cols=1  deg=6  hc=46,683,325  inner_hc=169  n_inners=1  context='On April 4, 2008, Beyo...'
    rows=13  n_cols=1  deg=6  hc=42,955,968  inner_hc=529  n_inners=1  context='Forbes magazine began ...'
    rows=13  n_cols=1  deg=3  hc=38,106,288  inner_hc=64  n_inners=1  context='In the spring of 1834,...'
    rows=9  n_cols=1  deg=4  hc=35,989,128  inner_hc=49  n_inners=1  con

## BIRD

In [10]:
show_analysis(all_results["BIRD"])

───────────────────────────────────────────────────────
BIRD  (3,509 blocks total, 0.1s)
  trivial (1-col)       : 38
  inner removed         : 0
  vertices (kept)       : 3,509
  edges (normal overlap): 62
  connected components  : 3,448
  component size dist   : {6: 1, 5: 1, 4: 3, 3: 15, 2: 13, 1: 3415}

  Top 5 components by total HC:

  [1] 1 blocks  total_hc=1,067,989,712  total_rows=17  max_rows=17  max_degree=0
  top 1 by hc:
    rows=17  n_cols=3  deg=0  hc=1,067,989,712  inner_hc=0  n_inners=0  Body='<p><strong>(Very) shor...' | PostDate='2011-02-22 02:08:30.0' | +[PostId]
  top 1 by inner_hc:
    rows=17  n_cols=3  deg=0  hc=1,067,989,712  inner_hc=0  n_inners=0  Body='<p><strong>(Very) shor...' | PostDate='2011-02-22 02:08:30.0' | +[PostId]
  top 1 by hc + inner_hc:
    rows=17  n_cols=3  deg=0  hc=1,067,989,712  inner_hc=0  n_inners=0  Body='<p><strong>(Very) shor...' | PostDate='2011-02-22 02:08:30.0' | +[PostId]

  [2] 1 blocks  total_hc=725,572,526  total_rows=8  max_row